# DIY Copilot
## Lesson: Coding Agents

Until now, our assistant answered **questions**: one request, maybe a few tool calls, one answer. Modern coding assistants – Claude Code, GitHub Copilot in agent mode, Cursor – can do more: you give them a **task** ("fix bug WORK-412"), and they work on it on their own. They read the code, change files, run the tests, read the errors, change the code again – until the task is done. This is a **coding agent**.

Technically, an agent is the tool loop you know from the function calling lesson – with three differences:

```
           ┌─────────────────────────────────────────────┐
 task ───► │ model: plan the next step                   │
           │   │                                         │
           │   ▼                                         │
           │ harness: check & run the tool               │◄── approval rules
           │   (read, search, edit, run tests)           │
           │   │                                         │
           │   ▼                                         │
           │ model: look at the result ── not done? ─────┘
           │   │
           │   ▼ done
 result ◄──┘ summary + changes (diff)
```

1. **A goal instead of a question.** The agent needs a condition it can check itself: *all tests pass*. The tests are the most important feedback of a coding agent.
2. **Tools that change things.** The agent edits files and runs code – the harness must decide what it may do without asking (**approval**).
3. **Many steps.** A task easily takes 20 model calls – everything from the context engineering lesson applies.

During this lesson you will learn:
* which tools a coding agent needs and how they are built
* how **project instructions** (`AGENTS.md`) tell the agent how a repository works
* how an agent fixes a bug on its own – and why "all tests pass" is not enough
* how **approval rules** keep the agent within limits
* how agents find their way through a code base
* how an agent **framework** (Google ADK) does the same with less code

>**Free tier:** This notebook makes about 30 LLM requests. An agent run takes a few minutes, because the free tier allows only 15 requests per minute.

### Set up the environment

In [ ]:
# >>> SETUP <<<
# In Google Colab, this cell
#   1. clones the workshop repository,
#   2. installs its dependencies into the Python environment of this notebook (with uv),
#   3. makes the shared code (lib) importable.
# Locally (after `uv sync` in the repository), only step 3 is needed.
import subprocess
import sys
from pathlib import Path

REPO_URL = "https://github.com/openknowledge/diy-copilot-bootcamp"
IN_COLAB = "google.colab" in sys.modules

if IN_COLAB:
    REPO_DIR = Path("/content/diy-copilot")
    if not REPO_DIR.exists():
        subprocess.run(["git", "clone", "--depth", "1", REPO_URL, str(REPO_DIR)], check=True)
    subprocess.run([sys.executable, str(REPO_DIR / "setup" / "install_dependencies.py")], check=True)
else:
    # The repository root is the first parent directory containing the shared code
    REPO_DIR = next(p for p in Path.cwd().resolve().parents if (p / "lib").is_dir() and (p / "pyproject.toml").exists())

# The repository is not an installed package, so we add it to the module search path
if str(REPO_DIR) not in sys.path:
    sys.path.insert(0, str(REPO_DIR))
print(f"Workshop repository: {REPO_DIR}")

In [ ]:
import tempfile
from pathlib import Path

from google.adk.agents import Agent
from google.adk.models.lite_llm import LiteLlm

from lib.coding_agent import (
    WORKSPACE,
    adk_approval_callback,
    check_meeting_titles_not_logged,
    count_tests,
    create_workspace,
    edit_file,
    list_files,
    quiet_adk,
    read_file,
    repo_map,
    run_adk_agent,
    run_agent,
    run_tests,
    search_code,
    show_diff,
)
from lib.context_engineering import count_tokens
from lib.environment import load_environment

default_models = load_environment()
generation_model = default_models["generation_model"]
print(f"Generation model: {generation_model}")

### Example: The workspace and the tools

The agent must not work on the original code. It gets a **workspace**: a copy of the Roomie repository in a temporary folder. Real coding agents work in your repository – but they show you every change, and you can undo it with git.

In [ ]:
create_workspace()
print(f"Workspace: {WORKSPACE}\n")
print(list_files())

The agent has five tools. They are normal Python functions – you can call them yourself:

| Tool | What it does |
|---|---|
| `list_files(directory)` | lists the files of the repository |
| `read_file(path)` | returns the content of a file |
| `search_code(text)` | finds all lines that contain a text – like `grep` |
| `edit_file(path, old_text, new_text)` | replaces a piece of text in a file |
| `run_tests()` | runs all tests with `pytest` |

In [ ]:
print(search_code("def check_"))

In [ ]:
print(run_tests())

`edit_file` doesn't rewrite the whole file. The model only sends the piece of text to replace and the new text – that saves tokens and avoids that the model accidentally changes other parts of the file. Most coding agents edit files this way. The text must be **unique** – otherwise the tool returns an error message, and the model can try again with more surrounding lines:

In [ ]:
print(edit_file("roomie/config.py", old_text="15", new_text="30"))

### Exercise 01: Project instructions (`AGENTS.md`)

Every repository has its own rules: how to run the tests, where things belong, what must never happen. Coding agents read them from a file in the repository root and put them into the system prompt of every request:

* `AGENTS.md` – an open format supported by many agents (e.g. GitHub Copilot, Cursor, OpenAI Codex)
* `CLAUDE.md` – Claude Code
* `.github/copilot-instructions.md` – GitHub Copilot

The file belongs to the **stable** part of the context – it is cached (see the context engineering lesson). Keep it short: only what the agent can't find out itself.

Your task is to write the `AGENTS.md` for Roomie. Useful topics:

* how to check the work (the tool `run_tests`) and when a task is done
* the rules for bug fixes and tests (see the testing guidelines: TEST-04, TEST-07)
* how new errors are added (see `docs/error_codes.md`)
* what the agent must never do

>**Reminder:** `%%writefile` saves the cell as a file (see the context engineering lesson).

In [ ]:
AGENTS_FILE = Path(tempfile.gettempdir()) / "AGENTS.md"

In [ ]:
%%writefile {AGENTS_FILE}
# Roomie – instructions for coding agents

## Checking your work
- Run the tests with the tool run_tests. A task is only done when all tests pass.
- When all tests pass, stop and answer with a short summary of your changes.

## Code
- Business rules live in roomie/services/booking_rules.py, never in the routers.
- New errors: a subclass of RoomieError in roomie/errors.py with status_code and code,
  plus a row in the table of docs/error_codes.md.

## Tests
- Every bug fix needs a regression test that reproduces the bug (TEST-07). Add it as a new
  test function; its docstring starts with "Regression test for <ticket id>."
- API error tests assert the error code from the response body, not only the status (TEST-04).
- Never change or delete existing tests to make them pass. Only add new tests.

The harness puts your instructions behind its own, general instructions for the agent:

In [ ]:
AGENT_INSTRUCTIONS = (
    "You are a coding agent working on the Roomie repository, the meeting room booking service of Team Workplace. "
    "Work on your own with the tools until the task is done: explore the code, make small and focused changes, "
    "run the tests and fix problems. Don't ask the developer - they are not available."
)
SYSTEM_PROMPT = f"{AGENT_INSTRUCTIONS}\n\n## Project instructions (AGENTS.md)\n\n{AGENTS_FILE.read_text()}"
print(SYSTEM_PROMPT)

### Exercise 02: The agent fixes a bug

Remember the session of the context engineering lesson: bug `WORK-412`, and the team decided how to fix it. Now the agent does the work. Run the cell and watch the steps – every line is one tool call.

>**Note:** This takes a few minutes and about 15–20 requests. The agent is not deterministic: your run will look different from your neighbour's.

In [ ]:
WORK_412_TASK = (
    "Fix bug WORK-412: POST /api/v2/bookings answers with 500 instead of 422 if start or end are sent without "
    "timezone offset (e.g. '2026-03-03T10:00:00'). Decision of the team: reject such timestamps with the new error "
    "code ROOM-422-NAIVE-TIME (exception class NaiveTimestampError, status 422). Add the check as the first rule in "
    "validate_new_booking, in a new function check_timezone_aware."
)

create_workspace()
tests_before = count_tests()

answer = run_agent(SYSTEM_PROMPT, WORK_412_TASK, model=generation_model)
print(f"\n{answer}")

The agent says it's done. Can we trust it? Your task is to **check the result** – like a reviewer:

1. Do all tests pass? Run them yourself – don't rely on the agent's summary.
2. Did the agent add a test – and didn't remove one? Compare the number of tests with `tests_before`.
3. Read the **diff**: is the change what the team decided? Did the agent change anything it shouldn't have?

In [ ]:
# TODO: Check the result of the agent
print(run_tests())

tests_after = count_tests()
print(f"Tests before: {tests_before}, after: {tests_after}")
print(f"{'✅' if tests_after > tests_before else '❌'} At least one new test, no test removed")

show_diff()

What did we learn?
* The agent found the right places in the code on its own, changed several files and used the tests to check its work – with nothing but the five tools and a loop.
* "All tests pass" is a good stop condition, but not a proof. In one of our test runs, the agent **replaced an existing test** by its new one – the tests were green, the summary said "all tests pass", although `AGENTS.md` forbids exactly that.
* That's why every coding agent shows you the diff, and why the harness should check what it can check deterministically (e.g. the number of tests) instead of trusting the model.

### Exercise 03: Approval rules

Real coding agents don't let the model do everything. They have **approval rules**: reading is fine, but some actions need the approval of the developer – e.g. editing files, running commands, or anything outside the project. In the function calling lesson you saw a simple version ("human in the loop").

Your task is to write the rule for our agent: the function `approve` gets the name of the tool and its arguments and returns `True` (allowed) or `False` (not allowed).

* Reading and searching is always allowed.
* Running the tests is allowed – the agent works in a copy.
* Editing files in `roomie/` and `docs/` is allowed.
* Editing files in `tests/` is only allowed if no existing code is removed: `old_text` must still be part of `new_text`.
* Everything else is not allowed.

>**Tip:** `"abc" in "xabcx"` checks whether a text is part of another text. `path.startswith("tests/")` checks the beginning of a text.

In [ ]:
# TODO: Implement the approval rules
def approve(tool_name: str, arguments: dict) -> bool:
    if tool_name in ("list_files", "read_file", "search_code", "run_tests"):
        return True
    if tool_name == "edit_file":
        path = arguments["path"]
        if path.startswith("roomie/") or path.startswith("docs/"):
            return True
        if path.startswith("tests/"):
            return arguments["old_text"] in arguments["new_text"]
    return False

Let's test the rules with a few tool calls – without the model:

In [ ]:
examples = [
    ("read_file", {"path": "tests/conftest.py"}, True),
    ("edit_file", {"path": "roomie/errors.py", "old_text": "class DurationError", "new_text": "class NewError(RoomieError): ...\n\n\nclass DurationError"}, True),
    ("edit_file", {"path": "tests/test_bookings_api.py", "old_text": "    assert response.json()[\"status\"] == \"checked_in\"\n",
                   "new_text": "    assert response.json()[\"status\"] == \"checked_in\"\n\n\ndef test_new_rule(client):\n    ...\n"}, True),
    ("edit_file", {"path": "tests/test_bookings_api.py", "old_text": "def test_check_in_within_grace_period(client, clock):",
                   "new_text": "def test_naive_timestamp_rejected(client):"}, False),
    ("edit_file", {"path": "pyproject.toml", "old_text": "pytest>=8.0", "new_text": "pytest>=9.0"}, False),
    ("delete_repository", {}, False),
]
for tool_name, arguments, expected in examples:
    result = approve(tool_name, arguments)
    print(f"{'✅' if result == expected else '❌'} {tool_name}({arguments.get('path', '')}) -> {'allowed' if result else 'not allowed'}")

`run_agent` accepts the rule as a parameter (`approve=approve`). If a tool call is not approved, the agent is told so and has to find another way. We'll use the rule in exercise 04.

>**Note:** In a real coding agent, "not allowed" usually means "ask the developer". In a notebook, you could do that with `input()`: `return input(f"Allow {tool_name}? (y/n) ") == "y"`.

### Example: How do agents find their way through the code?

Our agent found the right files by listing, searching and reading. With 30 files that's easy – a real repository has thousands. Coding agents use different strategies (often combined):

| Strategy | How | Used e.g. by | Strengths and weaknesses |
|---|---|---|---|
| **Agentic search** | the agent searches with tools like `grep` and reads files – step by step | Claude Code | always up to date, exact for names; needs many steps |
| **Semantic index** | code chunks are embedded in a vector store, like the knowledge base on day 1 | Cursor | finds code by meaning ("where are bookings validated?"); the index must be kept up to date |
| **Repository map** | a compact overview of all files, classes and function signatures in the context | Aider | the agent sees the structure at a glance; costs tokens in every request |

Here is a repository map of Roomie, built with Python's `ast` module:

In [ ]:
overview = repo_map()
print(overview[:2000])
print(f"\n... about {count_tokens(overview)} tokens for the whole map")

### Exercise 04: The same agent with a framework

We built the agent loop ourselves to understand it. In practice, you would use an **agent framework**: Google ADK, the OpenAI Agents SDK, the Claude Agent SDK, LangGraph, ... They all contain the same loop – plus sessions, streaming, sub-agents, tracing and more.

With [Google ADK](https://google.github.io/adk-docs/), an agent consists of:

* `model` – which model to use (via LiteLLM, like in all lessons)
* `instruction` – the system prompt
* `tools` – a list of Python functions. ADK reads the **docstring** and the type hints and generates the tool declarations itself – like FastMCP in the MCP lesson.
* `before_tool_callback` – a function that ADK calls before every tool call. That's where approval rules go.

The tools are the same functions as before – look at the docstring ADK will read:

In [ ]:
help(edit_file)

This time the agent gets another task: the security review found that `roomie/notifications.py` writes **meeting titles** into the log – that violates SEC-04 of our security requirements (no personal data beyond the user id in logs).

Your task is to create the agent with your system prompt, the five tools and your approval rule. `adk_approval_callback(approve)` turns your `approve` function into a callback for ADK.

In [ ]:
quiet_adk()
create_workspace()
tests_before = count_tests()

# TODO: Create the agent with the system prompt, the tools and the approval rule
agent = Agent(
    name="roomie_agent",
    model=LiteLlm(model=generation_model, num_retries=6),
    instruction=SYSTEM_PROMPT,
    tools=[list_files, read_file, search_code, edit_file, run_tests],
    before_tool_callback=adk_approval_callback(approve),
)

SEC_04_TASK = (
    "The security review found that roomie/notifications.py writes meeting titles into the log. That violates SEC-04: "
    "logs must not contain personal data beyond the user id, and meeting titles are personal data. "
    "Fix it: the notifications themselves must still contain the title, only the log must not."
)
answer = await run_adk_agent(agent, SEC_04_TASK)
print(f"\n{answer}")

Check the result again. This time we also have an independent check: it sends notifications for a meeting called "Performance review with Anna" and looks into the log.

In [ ]:
print(run_tests())
tests_after = count_tests()
print(f"{'✅' if tests_after >= tests_before else '❌'} No test removed (before: {tests_before}, after: {tests_after})")
check_meeting_titles_not_logged()
show_diff()

What did we learn?
* The framework hides the loop we built ourselves: calling the model, executing the tools, sending the results back, stopping. The concepts stay the same – system prompt, tool descriptions, approval rules.
* The tool declarations come from the docstrings. Docstrings are prompts – write them carefully.
* Frameworks make it easy to start, but they also hide what is sent to the model. When an agent behaves strangely, you need to understand the loop underneath – which you do now.

### Learnings
* A coding agent is the tool loop with a **goal**: it works on its own until a checkable condition is met. For code, the tests are the most important feedback.
* A few generic tools are enough: list, read, search, edit, run tests. Editing by "replace this text" instead of rewriting files is cheaper and safer.
* **Project instructions** (`AGENTS.md`, `CLAUDE.md`, ...) tell the agent how a repository works. Keep them short and stable.
* **Don't trust the summary** – check the result: run the tests, compare what you can count, read the diff. Agents can "fix" tests instead of code.
* **Approval rules** keep the agent within limits: read freely, ask before changing things or running commands, never act outside the project. Combined with a sandbox (our workspace), mistakes stay harmless.
* Agents find code by searching, by a semantic index or by a repository map – the same retrieval ideas as on day 1.
* Long-running agents need everything from the context engineering lesson – and more: **notes** as external memory (e.g. a to-do list in a file), and **sub-agents** that work on a part of the task in their own context and only return a summary.
* How good is a coding agent? Benchmarks like *SWE-bench* do what you did: give the agent real bug reports and check whether the tests pass afterwards.
* Frameworks such as Google ADK contain the same loop – and you now know what happens inside.